In [1]:
import sys
import os

# Add the src directory to sys.path
sys.path.append(os.path.abspath("../src"))

In [19]:
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
from typing import List, Tuple, Type

from abstractions3d.primitives.shapes import Box3D
from abstractions3d.primitives.visualization import visualize_boxes_3d
from abstractions3d.dsl.core import Shape3D
from abstractions3d.dsl.nodes import Rect3D, Move3D, Union3D, SymTrans3D, SymRef3D
from abstractions3d.dsl.instantiation import instantiate_3d

In [29]:
class AbstractionNode(Shape3D):
    def __init__(self, type_list: list, latent: torch.Tensor, model: nn.Module):
        super().__init__(children=[])
        self.type_list = type_list
        self.latent = latent
        self.model = model

    def expand(self) -> Shape3D:
        if self.model is None:
            param_list = self.latent.tolist()
        else:
            self.model.eval()
            with torch.no_grad():
                param_list = self.model.decoder(self.latent[None, :])[0].tolist()
        return instantiate_3d(self.type_list, param_list)

    def get_box3d_list(self):
        return self.expand().get_box3d_list()

    @classmethod
    def from_shape(cls, shape: Shape3D, models: dict):
        type_list, numeric_params = [], []

        def collect(s):
            type_list.append(type(s))
            for p in s.param_tuple()[1]:
                if isinstance(p, (float, int)):
                    numeric_params.append(p)
            for c in getattr(s, 'children', []):
                collect(c)

        collect(shape)
        model = models.get(type(shape).__name__)
        latent = torch.tensor(numeric_params, dtype=torch.float32)
        return cls(type_list, latent, model)

In [30]:
def add(structures1, structures2):
    for k in structures2:
        structures1[k] += structures2[k]
    return structures1

def get_singletons(shapes):
    singletons = defaultdict(list)
    if isinstance(shapes, list):
        for s in shapes:
            singletons = add(singletons, get_singletons(s))
        return singletons

    t, params = shapes.param_tuple()
    floats = tuple(p for p in params if isinstance(p, (float,int)))
    if floats:
        singletons[t.__name__].append(floats)

    for c in getattr(shapes, 'children', []):
        singletons = add(singletons, get_singletons(c))
    return singletons

def get_pairs(shapes):
    pairs = defaultdict(list)
    if isinstance(shapes, list):
        for s in shapes:
            pairs = add(pairs, get_pairs(s))
        return pairs

    if isinstance(shapes, Union3D):
        t, (c1, c2) = shapes.param_tuple()
        t1, p1 = c1.param_tuple()
        t2, p2 = c2.param_tuple()
        key = f"{t.__name__}({t1.__name__},{t2.__name__})"
        pairs[key].append(tuple([p for p in p1 if isinstance(p,(float,int))] +
                                [p for p in p2 if isinstance(p,(float,int))]))
        for c in shapes.children:
            pairs = add(pairs, get_pairs(c))
    else:
        for c in getattr(shapes, 'children', []):
            pairs = add(pairs, get_pairs(c))
    return pairs

In [31]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(), nn.Linear(32, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

def prepare_autoencoder_train_data(parameters, mask):
    tensor = torch.tensor(parameters, dtype=torch.float32).T
    masked_data = tensor[mask]
    dataset = TensorDataset(masked_data)
    return DataLoader(dataset, batch_size=64, shuffle=True)

def is_well_explained(parameters, model, threshold):
    model.eval()
    with torch.no_grad():
        _, recon = model(parameters)
        error,_ = torch.max(torch.abs(recon - parameters), dim=-1)
        return error < threshold

In [32]:
def find_abstractions(structures, retrain_iterations=2, error_threshold=0.01):
    models = {}
    losses = defaultdict(list)

    for name, parameters in structures.items():
        float_indices = [i for i, p in enumerate(parameters[0]) if isinstance(p, float)]
        if len(float_indices) == 0:
            continue
        float_params = [[p[i] for i in float_indices] for p in parameters]
        num_float_parameters = len(float_indices)
        well_explained = torch.ones(len(float_params[0]), dtype=torch.bool)

        for _ in range(retrain_iterations):
            train_tensor = torch.tensor(float_params, dtype=torch.float32)
            train_dl = DataLoader(TensorDataset(train_tensor), batch_size=64, shuffle=True)
            model = Autoencoder(input_dim=num_float_parameters, latent_dim=max(1,num_float_parameters-1))
            optimizer = AdamW(model.parameters(), lr=1e-3)
            loss_fn = lambda pred, target: torch.mean(torch.max(torch.abs(pred-target), dim=-1)[0])
            model.train()
            for batch in train_dl:
                x = batch[0]
                optimizer.zero_grad()
                _, x_rec = model(x)
                loss = loss_fn(x_rec, x)
                loss.backward()
                optimizer.step()
            with torch.no_grad():
                _, recon = model(train_tensor)
                error,_ = torch.max(torch.abs(recon - train_tensor), dim=-1)
                well_explained = error < error_threshold
        models[name] = model
    return models, losses

In [33]:
# Chair
seat = Rect3D(2.0,0.5,2.0)
back = Move3D(Rect3D(2.0,2.0,0.5),0,1.25,-0.75)
leg1 = Move3D(Rect3D(0.2,1.0,0.2),-0.9,-0.75,0.9)
leg2 = Move3D(Rect3D(0.2,1.0,0.2),0.9,-0.75,0.9)
leg3 = Move3D(Rect3D(0.2,1.0,0.2),-0.9,-0.75,-0.9)
leg4 = Move3D(Rect3D(0.2,1.0,0.2),0.9,-0.75,-0.9)

chair = Union3D(seat, back)
chair = Union3D(chair, leg1)
chair = Union3D(chair, leg2)
chair = Union3D(chair, leg3)
chair = Union3D(chair, leg4)

# Extract structures
singletons = get_singletons([chair])
pairs = get_pairs([chair])
structures = add(singletons, pairs)

# Train models
models, losses = find_abstractions(structures)

# Integrate abstractions
abstracted = AbstractionNode.from_shape(chair, models)

# Reconstruct and visualize
reconstructed = abstracted.expand()
boxes = reconstructed.get_box3d_list()
visualize_boxes_3d(boxes)

IndexError: pop from empty list

In [43]:
abstracted.expand()

IndexError: pop from empty list